# Data Types in Finance
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press

---
**Learning Objectives:**
- Distinguish time series, cross-sectional, and panel data
- Calculate simple and log returns
- Visualise and explore real financial data

> **How to use this notebook:** Run each cell in order (Shift+Enter). Try changing the ticker symbols or date ranges!


## Step 0 — Install & Import Libraries

In [ ]:
# Install yfinance (only needed in Colab)
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'; GREY = '#4B4B4B'
print('Libraries loaded ✓')

---
## Part 1 — Time Series Data
### 1.1 Download Stock Price Data

In [ ]:
# ── Download data ─────────────────────────────────────────────────
# Try changing the ticker! e.g. 'MSFT', 'TSLA', 'NESN.SW'
TICKER = 'AAPL'
START  = '2019-01-01'
END    = '2024-12-31'

data   = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
prices = data['Close'].squeeze()

print(f'Downloaded {TICKER}: {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'\nFirst 5 rows:')
print(prices.head())

### 1.2 Plot Price Series (Time Series)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.05})

# Price
axes[0].plot(prices.index, prices.values, color='black', lw=1.3)
axes[0].fill_between(prices.index, prices.values, alpha=0.06, color='black')
axes[0].set_ylabel('Price (USD)', fontsize=11)
axes[0].set_title(f'{TICKER} — Price & Daily Returns ({START[:4]}–{END[:4]})',
                  fontsize=13, fontweight='bold')

# Daily returns
ret = prices.pct_change().dropna() * 100
colors = [RED if v < 0 else 'black' for v in ret.values]
axes[1].bar(ret.index, ret.values, color=colors, width=1.0, alpha=0.7)
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_ylabel('Daily return (%)', fontsize=11)
axes[1].set_xlabel('Date', fontsize=11)

plt.tight_layout()
plt.show()
print('→ This is TIME SERIES data: one entity (stock), many time points')

---
## Part 2 — Simple vs. Log Returns
### 2.1 Calculate Both Return Types

In [ ]:
simple_ret = prices.pct_change().dropna()
log_ret    = np.log(prices / prices.shift(1)).dropna()

# Align
idx = simple_ret.index.intersection(log_ret.index)
s = simple_ret.loc[idx] * 100
l = log_ret.loc[idx]    * 100
diff = s - l

print('Descriptive Statistics (in %)')
print('-' * 55)
stats = pd.DataFrame({'Simple return': s.describe(), 'Log return': l.describe()})
print(stats.round(4))
print(f'\nMax difference (simple − log): {diff.abs().max():.4f}%')
print(f'On date: {diff.abs().idxmax().date()}')

### 2.2 Visualise the Difference

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: daily returns overlay
ax = axes[0]
ax.plot(s.index, s.values, color='black', lw=0.8, alpha=0.8, label='Simple')
ax.plot(l.index, l.values, color=ORANGE, lw=0.8, alpha=0.8, ls='--', label='Log')
ax.axhline(0, color=GREY, lw=0.5)
ax.set_title('A: Daily Returns — Nearly Identical', fontsize=11, fontweight='bold')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9)
ax.text(0.02, 0.02, 'For small daily moves:\nlog ≈ simple',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(facecolor=YELLOW, alpha=0.85, edgecolor='none', boxstyle='round,pad=0.3'))

# Panel B: cumulative returns (the real difference!)
ax = axes[1]
cum_simple = (1 + simple_ret).cumprod() - 1
cum_log    = log_ret.cumsum()              # log returns are additive
ax.plot(cum_simple.index, cum_simple.values * 100, color='black', lw=1.5, label='Cumulative simple')
ax.plot(cum_log.index,    cum_log.values * 100,    color=RED,     lw=1.5, ls='--', label='Sum of log')
ax.set_title('B: Cumulative — Diverge Over Time!', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative return (%)')
ax.legend(fontsize=9)
ax.text(0.02, 0.97, '← Diverge here\n   (strong bull run)',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(facecolor='#FFE0E0', alpha=0.9, edgecolor='none', boxstyle='round,pad=0.3'))

# Panel C: difference = simple − log (shows where it matters)
ax = axes[2]
ax.fill_between(diff.index, diff.values, color=RED, alpha=0.5)
ax.axhline(0, color=GREY, lw=0.8)
ax.set_title('C: Difference (Simple − Log)', fontsize=11, fontweight='bold')
ax.set_ylabel('Difference (%)')
# annotate max divergence
max_idx = diff.abs().idxmax()
ax.annotate(f'Max: {diff[max_idx]:.2f}%\n({max_idx.date()})',
            xy=(max_idx, diff[max_idx]),
            xytext=(max_idx, diff[max_idx] * 0.4),
            arrowprops=dict(arrowstyle='->', color=RED),
            fontsize=9, color=RED, fontweight='bold')

plt.suptitle(f'Simple vs. Log Returns — {TICKER} {START[:4]}–{END[:4]}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key insight:')
print('• Daily returns: simple ≈ log (difference < 0.01%)')
print('• Cumulative returns: simple > log (diverge in bull markets)')
print('• Rule of thumb: log for modelling/statistics, simple for reporting')

### 2.3 Concrete Numerical Example
Price drops from \$100 to \$70 (−30%), then recovers to \$100 (+42.86%)

In [ ]:
print('=' * 60)
print('CONCRETE EXAMPLE: Crash and Recovery')
print('=' * 60)
P0, P1, P2 = 100, 70, 100

R1_s = (P1 - P0) / P0;         R2_s = (P2 - P1) / P1
R1_l = np.log(P1 / P0);        R2_l = np.log(P2 / P1)

print(f'\nDay 1: ${P0} → ${P1}')
print(f'  Simple return: {R1_s:.4f} = {R1_s*100:.2f}%')
print(f'  Log return:    {R1_l:.4f} = {R1_l*100:.2f}%')
print(f'\nDay 2: ${P1} → ${P2}')
print(f'  Simple return: {R2_s:.4f} = {R2_s*100:.2f}%')
print(f'  Log return:    {R2_l:.4f} = {R2_l*100:.2f}%')
print(f'\nTotal (simple): {(1+R1_s)*(1+R2_s)-1:.4f} = {((1+R1_s)*(1+R2_s)-1)*100:.2f}%  ← multiply')
print(f'Total (log):    {R1_l + R2_l:.4f} = {(R1_l+R2_l)*100:.2f}%  ← just add!')
print(f'\n→ Log returns are additive: {R1_l:.4f} + {R2_l:.4f} = {R1_l+R2_l:.4f} ≈ 0')
print(f'→ This is why log returns are preferred for multi-period analysis')

---
## Part 3 — Cross-Sectional & Panel Data
### 3.1 Download Multiple Assets (Panel)

In [ ]:
# Panel: multiple assets over time
tickers = ['AAPL', 'MSFT', 'JPM', '^GSPC']
raw     = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)
prices_panel = raw['Close'].copy()
prices_panel.columns = ['AAPL', 'MSFT', 'JPM', 'SP500']
prices_panel = prices_panel.dropna()

ret_panel = prices_panel.pct_change().dropna()
print(f'Panel shape: {ret_panel.shape}  →  {ret_panel.shape[0]} days × {ret_panel.shape[1]} assets')
print(f'\nData type: PANEL (multiple entities, multiple time points)')
print(f'\nSummary statistics:')
print((ret_panel * 100).describe().round(4))

### 3.2 Cross-Section: Beta for Each Stock on a Single Date

In [ ]:
from scipy import stats

# Estimate beta for each stock vs S&P 500
betas = {}
for col in ['AAPL', 'MSFT', 'JPM']:
    slope, intercept, r, p, se = stats.linregress(ret_panel['SP500'], ret_panel[col])
    betas[col] = {'Beta': round(slope, 3),
                  'Alpha (daily)': f"{intercept*100:.4f}%",
                  'R²': round(r**2, 3)}

beta_df = pd.DataFrame(betas).T
print('Cross-Sectional Beta Estimates (2019–2024)')
print('-' * 50)
print(beta_df)
print('\n→ This is CROSS-SECTIONAL: one beta per firm, estimated at one point in time')
print('→ Beta > 1: stock amplifies market moves (aggressive)')
print('→ Beta < 1: stock dampens market moves (defensive)')

### 3.3 Panel Visualisation: Cumulative Returns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative returns
cum = (1 + ret_panel).cumprod() - 1
colors_line = ['black', ORANGE, '#0E75FE', RED]
ax = axes[0]
for col, c in zip(cum.columns, colors_line):
    ax.plot(cum.index, cum[col] * 100, lw=1.6, color=c, label=col)
ax.axhline(0, color=GREY, lw=0.8)
ax.set_title('Cumulative Returns 2019–2024 (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Cumulative return (%)')
ax.legend(fontsize=10)

# Monthly heatmap
try:
    import seaborn as sns
    monthly = ret_panel.resample('ME').apply(lambda x: (1+x).prod()-1)
    mp = monthly.iloc[-24:] * 100
    mp.index = mp.index.strftime('%b %y')
    ax2 = axes[1]
    sns.heatmap(mp.T, ax=ax2, cmap='RdYlGn', center=0,
                annot=True, fmt='.0f', linewidths=0.5, linecolor='white',
                annot_kws={'size': 8},
                cbar_kws={'label': 'Monthly return (%)'})
    ax2.set_title('Monthly Returns Heatmap (last 24 months)', fontsize=12, fontweight='bold')
    plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
except ImportError:
    axes[1].text(0.5, 0.5, 'pip install seaborn for heatmap', ha='center', va='center')

plt.tight_layout()
plt.show()

---
## Part 4 — Annualising Statistics

In [ ]:
TRADING_DAYS = 252

print('Annualised Statistics (2019–2024)')
print('=' * 60)
for col in ret_panel.columns:
    daily_ret = ret_panel[col]
    ann_return = (1 + daily_ret.mean()) ** TRADING_DAYS - 1
    ann_vol    = daily_ret.std() * np.sqrt(TRADING_DAYS)
    sharpe     = ann_return / ann_vol  # simplified (no risk-free rate)
    print(f'{col:<8} | Annual return: {ann_return*100:6.1f}%  '
          f'| Annual vol: {ann_vol*100:5.1f}%  '
          f'| Sharpe (approx): {sharpe:.2f}')

print('\nFormulas:')
print('  Annual return = (1 + daily_mean)^252 − 1')
print('  Annual vol    = daily_std × √252')

---
## 🎯 Self-Check Exercises

Try solving these by modifying the code above:

1. **Change the ticker** to `NESN.SW` (Nestlé) or `TSLA`. How does beta compare to AAPL?
2. **Change the date range** to `2020-01-01` to `2020-12-31`. What do you observe during COVID?
3. **Calculate the worst single day** for each stock: `ret_panel.min()`
4. **Is daily vol of AAPL > JPM?** Check with `ret_panel.std() * np.sqrt(252)`

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*